In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install pytorch-lightning timm torchmetrics scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.2 MB/s eta 0:00:00


In [7]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/finetune_train.py
import os
import torch
import torch.nn as nn
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torchmetrics.classification import BinaryFBetaScore, BinaryRecall

from config import Config
from data_loaders import HAM10000Dataset
from models import GatedHybridCBM
from engine import CBM_System

# OVERRIDE ENGINE: Forcing Recall > Precision
class Finetune_CBM_System(CBM_System):
    def __init__(self, model, config):
        super().__init__(model, config)

        self.val_f2 = BinaryFBetaScore(beta=2.0)
        self.val_recall = BinaryRecall()

    def training_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch

        # Forward Pass
        c_logits, y_logit, alpha = self.model(x)

        # The Nuclear Option: 5x Penalty for False Negatives (dynamically mapped to GPU)
        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))

        # Calculate ONLY the task loss. We do not care about concepts during this boundary shift.
        loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch
        c_logits, y_logit, alpha = self.model(x)

        y_prob = torch.sigmoid(y_logit).view(-1)
        y_true = y_true.view(-1)

        self.val_f2(y_prob, y_true)
        self.val_recall(y_prob, y_true)

        self.log("val_f2", self.val_f2, on_epoch=True, prog_bar=True)
        self.log("val_recall", self.val_recall, on_epoch=True, prog_bar=True)

        # Required to keep ModelCheckpoint from crashing if it expects val_loss
        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))
        val_loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))
        self.log("val_loss", val_loss, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        # Explicitly filter out frozen parameters to prevent optimizer crashing
        trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
        return torch.optim.AdamW(trainable_params, lr=1e-5, weight_decay=1e-4)


def main():
    pl.seed_everything(42)
    print("IGNITING PHASE 1: Metric Re-Alignment & Boundary Shifting...")

    HAM_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/HAM10000"
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    meta_files = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')]
    if not meta_files: raise FileNotFoundError("Could not find metadata CSV in HAM10000 directory.")
    META_CSV = os.path.join(HAM_DIR, meta_files[0])

    print("🧠 Retrieving Release Candidate 1 (Epoch 20)...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    ckpt_path = os.path.join(CHECKPOINT_DIR, "ham-gated-cbm-epoch=20-val_auc=0.950.ckpt")
    base_system = CBM_System.load_from_checkpoint(ckpt_path, model=base_model, config=Config)
    system = Finetune_CBM_System(model=base_system.model, config=Config)

    # INVERTED DEEP FREEZE
    print("Freezing Backbone & Concepts. Unlocking Diagnostic Head...")
    frozen_count = 0
    trainable_count = 0
    for name, param in system.model.named_parameters():
        # Target the massive feature extractors for freezing
        if 'backbone' in name.lower() or 'concept' in name.lower():
            param.requires_grad = False
            frozen_count += 1
        else:
            param.requires_grad = True
            trainable_count += 1

    print(f"Locked {frozen_count} visual/concept tensors.")
    print(f"Unlocked {trainable_count} routing/classification tensors.")

    full_df = pd.read_csv(META_CSV)
    clean_dx = full_df['dx'].astype(str).str.lower().str.strip()
    binary_labels = (clean_dx == 'mel').astype(int)

    train_df, val_df = train_test_split(full_df, test_size=0.2, stratify=binary_labels, random_state=42)

    train_labels = (train_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    weight_melanoma = 1.0 / train_labels.sum()
    weight_other = 1.0 / (len(train_df) - train_labels.sum())
    sample_weights = [weight_melanoma if l == 1 else weight_other for l in train_labels]

    sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

    train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    val_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    # --- RE-INJECTED DATASETS ---
    train_dataset = HAM10000Dataset(metadata_df=train_df, image_dir=HAM_DIR, transforms=train_transform)
    val_dataset = HAM10000Dataset(metadata_df=val_df, image_dir=HAM_DIR, transforms=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="safety-f2-cbm-{epoch:02d}-{val_f2:.3f}",
        monitor="val_f2",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_f2",
        min_delta=0.005,
        patience=3,
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_Safety")

    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("Executing Decision Boundary Shift...")
    trainer.fit(system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"Metric Re-Alignment Complete. Safe Model saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/finetune_train.py


In [8]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python finetune_train.py

Streaming output truncated to the last 5000 lines.
                                                               train_loss: 0.656
                                                               val_f2: 0.707    
                                                               val_recall: 0.839
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 87/501 0:00:41 • 0:03:18 2.09it/s v_num: 1.000     
                                                               train_loss: 0.493
                                                               val_f2: 0.707    
                                                               val_recall: 0.839
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 88/501 0:00:41 • 0:03:15 2.12it/s v_num: 1.000     
                                                               train_loss: 0.470
                                                               val_f2: 0.707    
                                                               val_recall: 0.839
Epoch 2/9  ━━━╺━━━━━━━━━━━━━ 89/501 0:00:42 • 0:03:16 2.10

In [11]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/final_acid_test.py
import os
import torch
import pandas as pd
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from config import Config
from models import GatedHybridCBM
from engine import CBM_System

def execute_dual_audit():
    print("INITIALIZING FINAL DUAL TEST...")

    # 1. Paths
    BASE_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/"
    HAM_DIR = os.path.join(BASE_DIR, "HAM10000")
    ISIC_DIR = os.path.join(BASE_DIR, "isic_2020")
    CHECKPOINT_DIR = os.path.join(BASE_DIR, "cloud_checkpoints")

    # Target the NEW safety-aligned checkpoint
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, "safety-f2-cbm-epoch=00-val_f2=0.717.ckpt")

    # 2. Boot the Aligned Engine
    print(f"Injecting Safe Memory Overlay: {os.path.basename(CKPT_PATH)}...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    system = CBM_System.load_from_checkpoint(CKPT_PATH, model=base_model, config=Config)
    system.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    system.to(device)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # A: HAM10000 (In-Distribution Sanity Check)
    print("\nPrepping Cohort A: HAM10000 Validation Sample...")
    ham_meta = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')][0]
    ham_df = pd.read_csv(os.path.join(HAM_DIR, ham_meta))

    # We strictly use a validation split so the model is not tested on its own training data
    binary_labels = (ham_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    _, val_df = train_test_split(ham_df, test_size=0.2, stratify=binary_labels, random_state=42)

    # Sample 200 random images from the validation set for speed
    ham_sample = val_df.sample(n=200, random_state=42).reset_index(drop=True)

    ham_results = {'y_true': [], 'y_prob': [], 'alpha': []}

    with torch.no_grad():
        for _, row in tqdm(ham_sample.iterrows(), total=len(ham_sample), desc="Auditing HAM10k"):
            # HAM images might be in subfolders, dynamically find it
            img_id = str(row['image_id'])
            img_path = None
            for root, _, files in os.walk(HAM_DIR):
                if f"{img_id}.jpg" in files:
                    img_path = os.path.join(root, f"{img_id}.jpg")
                    break

            if img_path:
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                ham_results['y_true'].append(1 if str(row['dx']).lower().strip() == 'mel' else 0)
                ham_results['y_prob'].append(torch.sigmoid(y_logit).item())
                ham_results['alpha'].append(alpha.item())

    # CB: ISIC 2020 (OOD Micro-Batch)
    print("\nPrepping Cohort B: ISIC 2020 Micro-Batch...")
    isic_csv = os.path.join(ISIC_DIR, "train.csv")
    isic_img_dir = os.path.join(ISIC_DIR, "images")

    if os.path.exists(isic_csv) and os.path.exists(isic_img_dir):
        isic_df = pd.read_csv(isic_csv)
        uploaded_files = [f for f in os.listdir(isic_img_dir) if f.endswith(('.jpg', '.jpeg'))]
        uploaded_basenames = [os.path.splitext(f)[0] for f in uploaded_files]
        test_df = isic_df[isic_df['image_name'].isin(uploaded_basenames)].reset_index(drop=True)

        isic_results = {'y_true': [], 'y_prob': [], 'alpha': []}

        with torch.no_grad():
            for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Auditing ISIC"):
                img_path = os.path.join(isic_img_dir, str(row['image_name']) + ".jpg")
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                isic_results['y_true'].append(row['target'])
                isic_results['y_prob'].append(torch.sigmoid(y_logit).item())
                isic_results['alpha'].append(alpha.item())
    else:
        print("ISIC Data not found. Skipping B.")
        isic_results = None

    # THE VERDICT
    def print_report(name, res):
        print(f"\n" + "="*50)
        print(f"{name} AUDIT REPORT (POST-FINETUNE)")
        print("="*50)

        y_true = res['y_true']
        y_prob = res['y_prob']
        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        alphas = res['alpha']

        if len(set(y_true)) > 1:
            print(f"AUC Score: {roc_auc_score(y_true, y_prob):.3f}")

        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel() if len(cm.ravel()) == 4 else (0,0,0,0)

        print(f"\n--- SAFETY METRICS (Threshold 0.5) ---")
        print(f"Total Melanomas Present: {tp + fn}")
        print(f"FALSE NEGATIVES (Missed Cancers): {fn}")
        print(f"Sensitivity (Recall): {tp/(tp+fn) if (tp+fn)>0 else 0:.2f}")

        print(f"\n--- XAI INTEGRITY ---")
        print(f"Average Alpha Gate: {np.mean(alphas):.3f}")
        print(f"Black-Box Bias (<0.2): {sum(1 for a in alphas if a < 0.2) / len(alphas) * 100:.1f}%")

    print_report("A: HAM10000", ham_results)
    if isic_results:
        print_report("B: ISIC 2020", isic_results)

if __name__ == "__main__":
    execute_dual_audit()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/final_acid_test.py


In [12]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python final_acid_test.py

/content/drive/MyDrive/XAI_Cloud_Run
INITIALIZING FINAL DUAL TEST...
Injecting Safe Memory Overlay: safety-f2-cbm-epoch=00-val_f2=0.717.ckpt...

Prepping Cohort A: HAM10000 Validation Sample...
Auditing HAM10k: 100% 200/200 [00:24<00:00,  8.10it/s]

Prepping Cohort B: ISIC 2020 Micro-Batch...
Auditing ISIC: 100% 27/27 [00:05<00:00,  4.64it/s]

A: HAM10000 AUDIT REPORT (POST-FINETUNE)
AUC Score: 0.981

--- SAFETY METRICS (Threshold 0.5) ---
Total Melanomas Present: 17
FALSE NEGATIVES (Missed Cancers): 0
Sensitivity (Recall): 1.00

--- XAI INTEGRITY ---
Average Alpha Gate: 0.139
Black-Box Bias (<0.2): 69.5%

B: ISIC 2020 AUDIT REPORT (POST-FINETUNE)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(

--- SAFETY METRICS (Threshold 0.5) ---
Total Melanomas Present: 0
FALSE NEGAT

In [15]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/finetune_core.py
import os
import torch
import torch.nn as nn
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torchmetrics.classification import BinaryFBetaScore, BinaryRecall

from config import Config
from data_loaders import HAM10000Dataset
from models import GatedHybridCBM
from engine import CBM_System

#OVERRIDE ENGINE: Freeze
class Finetune_Core_System(CBM_System):
    def __init__(self, model, config):
        super().__init__(model, config)
        self.val_f2 = BinaryFBetaScore(beta=2.0)
        self.val_recall = BinaryRecall()

    def training_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch
        c_logits, y_logit, alpha = self.model(x)

        # 5x Penalty for False Negatives
        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))
        loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch
        c_logits, y_logit, alpha = self.model(x)

        y_prob = torch.sigmoid(y_logit).view(-1)
        y_true = y_true.view(-1)

        self.val_f2(y_prob, y_true)
        self.val_recall(y_prob, y_true)

        self.log("val_f2", self.val_f2, on_epoch=True, prog_bar=True)
        self.log("val_recall", self.val_recall, on_epoch=True, prog_bar=True)

        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))
        val_loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))
        self.log("val_loss", val_loss, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
        return torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

def main():
    pl.seed_everything(42)
    print("IGNITING PHASE 1 (REVISION): Absolute Freeze & Boundary Shifting...")

    HAM_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/HAM10000"
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    meta_files = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')]
    META_CSV = os.path.join(HAM_DIR, meta_files[0])

    print("Retrieving ORIGINAL Release Candidate...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    ckpt_path = os.path.join(CHECKPOINT_DIR, "ham-gated-cbm-epoch=20-val_auc=0.950.ckpt")
    base_system = CBM_System.load_from_checkpoint(ckpt_path, model=base_model, config=Config)
    system = Finetune_Core_System(model=base_system.model, config=Config)

    # EXCLUSION FREEZE
    print("Freezing Backbone, Concepts, AND Alpha Gate. Unlocking only the Final Classifier...")
    frozen_count = 0
    trainable_count = 0
    for name, param in system.model.named_parameters():
        # If it's the backbone, concept head, or the gate, lock it down.
        if 'backbone' in name.lower() or 'concept' in name.lower() or 'gate' in name.lower() or 'alpha' in name.lower():
            param.requires_grad = False
            frozen_count += 1
        else:
            # Whatever is left is the diagnostic classifier
            param.requires_grad = True
            trainable_count += 1

    print(f"\nLocked {frozen_count} visual/concept/gate tensors.")
    print(f"\nUnlocked {trainable_count} final classification tensors.")

    # Data Pipeline
    full_df = pd.read_csv(META_CSV)
    clean_dx = full_df['dx'].astype(str).str.lower().str.strip()
    binary_labels = (clean_dx == 'mel').astype(int)

    train_df, val_df = train_test_split(full_df, test_size=0.2, stratify=binary_labels, random_state=42)

    train_labels = (train_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    weight_melanoma = 1.0 / train_labels.sum()
    weight_other = 1.0 / (len(train_df) - train_labels.sum())
    sample_weights = [weight_melanoma if l == 1 else weight_other for l in train_labels]

    sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

    train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    val_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    train_dataset = HAM10000Dataset(metadata_df=train_df, image_dir=HAM_DIR, transforms=train_transform)
    val_dataset = HAM10000Dataset(metadata_df=val_df, image_dir=HAM_DIR, transforms=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="core-safe-cbm-{epoch:02d}-{val_f2:.3f}",
        monitor="val_f2",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_f2",
        min_delta=0.005,
        patience=3,
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_CoreSafe")

    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("Executing Surgical Boundary Shift...")
    trainer.fit(system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"Core Fixed. Safe Model saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/finetune_core.py


In [16]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python finetune_core.py

Streaming output truncated to the last 5000 lines.
                                                               val_f2: 0.689    
                                                               val_recall: 0.901
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 83/501 0:00:17 • 0:01:25 4.98it/s v_num: 1.000     
                                                               train_loss: 0.215
                                                               val_f2: 0.689    
                                                               val_recall: 0.901
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 84/501 0:00:17 • 0:01:24 4.98it/s v_num: 1.000     
                                                               train_loss: 0.268
                                                               val_f2: 0.689    
                                                               val_recall: 0.901
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 85/501 0:00:17 • 0:01:25 4.90it/s v_num: 1.000     
                                                          

In [17]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/final_acid_test.py
import os
import torch
import pandas as pd
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from config import Config
from models import GatedHybridCBM
from engine import CBM_System

def execute_dual_audit():
    print("INITIALIZING FINAL DUAL-COHORT ACID TEST...")

    BASE_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/"
    HAM_DIR = os.path.join(BASE_DIR, "HAM10000")
    ISIC_DIR = os.path.join(BASE_DIR, "isic_2020")
    CHECKPOINT_DIR = os.path.join(BASE_DIR, "cloud_checkpoints")

    # Target the NEW CORE-SAFE checkpoint
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, "core-safe-cbm-epoch=00-val_f2=0.712.ckpt")

    print(f"Injecting Core-Safe Memory Overlay: {os.path.basename(CKPT_PATH)}...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    system = CBM_System.load_from_checkpoint(CKPT_PATH, model=base_model, config=Config)
    system.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    system.to(device)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # ---------------------------------------------------------
    # COHORT A: HAM10000
    # ---------------------------------------------------------
    print("\nPrepping Cohort A: HAM10000 Validation Sample...")
    ham_meta = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')][0]
    ham_df = pd.read_csv(os.path.join(HAM_DIR, ham_meta))

    binary_labels = (ham_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    _, val_df = train_test_split(ham_df, test_size=0.2, stratify=binary_labels, random_state=42)

    ham_sample = val_df.sample(n=200, random_state=42).reset_index(drop=True)
    ham_results = {'y_true': [], 'y_prob': [], 'alpha': []}

    with torch.no_grad():
        for _, row in tqdm(ham_sample.iterrows(), total=len(ham_sample), desc="Auditing HAM10k"):
            img_id = str(row['image_id'])
            img_path = None
            for root, _, files in os.walk(HAM_DIR):
                if f"{img_id}.jpg" in files:
                    img_path = os.path.join(root, f"{img_id}.jpg")
                    break

            if img_path:
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                ham_results['y_true'].append(1 if str(row['dx']).lower().strip() == 'mel' else 0)
                ham_results['y_prob'].append(torch.sigmoid(y_logit).item())
                ham_results['alpha'].append(alpha.item())

    # ---------------------------------------------------------
    # COHORT B: ISIC 2020
    # ---------------------------------------------------------
    print("\nPrepping Cohort B: ISIC 2020 Micro-Batch...")
    isic_csv = os.path.join(ISIC_DIR, "train.csv")
    isic_img_dir = os.path.join(ISIC_DIR, "images")

    if os.path.exists(isic_csv) and os.path.exists(isic_img_dir):
        isic_df = pd.read_csv(isic_csv)
        uploaded_files = [f for f in os.listdir(isic_img_dir) if f.endswith(('.jpg', '.jpeg'))]
        uploaded_basenames = [os.path.splitext(f)[0] for f in uploaded_files]
        test_df = isic_df[isic_df['image_name'].isin(uploaded_basenames)].reset_index(drop=True)

        isic_results = {'y_true': [], 'y_prob': [], 'alpha': []}

        with torch.no_grad():
            for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Auditing ISIC"):
                img_path = os.path.join(isic_img_dir, str(row['image_name']) + ".jpg")
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                isic_results['y_true'].append(row['target'])
                isic_results['y_prob'].append(torch.sigmoid(y_logit).item())
                isic_results['alpha'].append(alpha.item())
    else:
        print("ISIC Data not found. Skipping Cohort B.")
        isic_results = None

    def print_report(name, res):
        print(f"\n" + "="*50)
        print(f"{name} AUDIT REPORT (CORE-SAFE)")
        print("="*50)

        y_true = res['y_true']
        y_prob = res['y_prob']
        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        alphas = res['alpha']

        if len(set(y_true)) > 1:
            print(f"AUC Score: {roc_auc_score(y_true, y_prob):.3f}")

        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel() if len(cm.ravel()) == 4 else (0,0,0,0)

        print(f"\n--- SAFETY METRICS (Threshold 0.5) ---")
        print(f"Total Melanomas Present: {tp + fn}")
        print(f"FALSE NEGATIVES (Missed Cancers): {fn}")
        print(f"Sensitivity (Recall): {tp/(tp+fn) if (tp+fn)>0 else 0:.2f}")

        print(f"\n--- XAI INTEGRITY ---")
        print(f"Average Alpha Gate: {np.mean(alphas):.3f}")
        print(f"Black-Box Bias (<0.2): {sum(1 for a in alphas if a < 0.2) / len(alphas) * 100:.1f}%")

    print_report("COHORT A: HAM10000", ham_results)
    if isic_results:
        print_report("COHORT B: ISIC 2020", isic_results)

if __name__ == "__main__":
    execute_dual_audit()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/final_acid_test.py


In [18]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python final_acid_test.py

/content/drive/MyDrive/XAI_Cloud_Run
INITIALIZING FINAL DUAL-COHORT ACID TEST...
Injecting Core-Safe Memory Overlay: core-safe-cbm-epoch=00-val_f2=0.712.ckpt...

Prepping Cohort A: HAM10000 Validation Sample...
Auditing HAM10k: 100% 200/200 [00:25<00:00,  7.74it/s]

Prepping Cohort B: ISIC 2020 Micro-Batch...
Auditing ISIC: 100% 27/27 [00:05<00:00,  4.64it/s]

COHORT A: HAM10000 AUDIT REPORT (CORE-SAFE)
AUC Score: 0.981

--- SAFETY METRICS (Threshold 0.5) ---
Total Melanomas Present: 17
FALSE NEGATIVES (Missed Cancers): 0
Sensitivity (Recall): 1.00

--- XAI INTEGRITY ---
Average Alpha Gate: 0.194
Black-Box Bias (<0.2): 50.5%

COHORT B: ISIC 2020 AUDIT REPORT (CORE-SAFE)

--- SAFETY METRICS (Threshold 0.5) ---
Total Melanomas Present: 0
FALSE NEGATIVES (Missed Cancers): 0
Sensitivity (Recall): 0.00

--- XAI INTEGRITY ---
Average Alpha Gate: 0.175
Black-Box Bias (<0.2): 55.6%


In [21]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/finetune_peak.py
import os
import torch
import torch.nn as nn
import pandas as pd
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torchmetrics.classification import BinaryFBetaScore, BinaryRecall

from config import Config
from data_loaders import HAM10000Dataset
from models import GatedHybridCBM
from engine import CBM_System

# ---------------------------------------------------------
# PEAK OVERRIDE ENGINE: Native Metric Maximization
# ---------------------------------------------------------
class Finetune_Peak_System(CBM_System):
    def __init__(self, model, config):
        super().__init__(model, config)
        self.val_f2 = BinaryFBetaScore(beta=2.0)
        self.val_recall = BinaryRecall()

    def on_train_epoch_start(self):
        super().on_train_epoch_start()
        # CRITICAL FIX: Force the entire architecture into eval mode.
        # This permanently locks BatchNorm running stats.
        # The optimizer will still update the weights that have requires_grad=True.
        self.model.eval()

    def training_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch
        c_logits, y_logit, alpha = self.model(x)

        # 5x Native Penalty for False Negatives
        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))
        loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, c_true, c_mask, y_true = batch
        c_logits, y_logit, alpha = self.model(x)

        y_prob = torch.sigmoid(y_logit).view(-1)
        y_true = y_true.view(-1)

        self.val_f2(y_prob, y_true)
        self.val_recall(y_prob, y_true)

        self.log("val_f2", self.val_f2, on_epoch=True, prog_bar=True)
        self.log("val_recall", self.val_recall, on_epoch=True, prog_bar=True)

        task_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=self.device))
        val_loss = task_loss_fn(y_logit.view(-1), y_true.float().view(-1))
        self.log("val_loss", val_loss, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
        return torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

def main():
    pl.seed_everything(42)
    print("INITIALIZING PEAK FORM: State Override & Native Boundary Shift...")

    HAM_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/HAM10000"
    CHECKPOINT_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/cloud_checkpoints/"

    meta_files = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')]
    META_CSV = os.path.join(HAM_DIR, meta_files[0])

    print("Retrieving PRISTINE Release Candidate (Epoch 20)...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    ckpt_path = os.path.join(CHECKPOINT_DIR, "ham-gated-cbm-epoch=20-val_auc=0.950.ckpt")
    base_system = CBM_System.load_from_checkpoint(ckpt_path, model=base_model, config=Config)
    system = Finetune_Peak_System(model=base_system.model, config=Config)

    print("Executing Bulletproof Exclusion Freeze...")
    frozen_count = 0
    trainable_count = 0
    for name, param in system.model.named_parameters():
        if 'backbone' in name.lower() or 'concept' in name.lower() or 'gate' in name.lower() or 'alpha' in name.lower():
            param.requires_grad = False
            frozen_count += 1
        else:
            param.requires_grad = True
            trainable_count += 1

    print(f"Locked {frozen_count} visual/concept/gate tensors.")
    print(f"Unlocked {trainable_count} final classification tensors.")

    full_df = pd.read_csv(META_CSV)
    clean_dx = full_df['dx'].astype(str).str.lower().str.strip()
    binary_labels = (clean_dx == 'mel').astype(int)

    train_df, val_df = train_test_split(full_df, test_size=0.2, stratify=binary_labels, random_state=42)

    train_labels = (train_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    weight_melanoma = 1.0 / train_labels.sum()
    weight_other = 1.0 / (len(train_df) - train_labels.sum())
    sample_weights = [weight_melanoma if l == 1 else weight_other for l in train_labels]

    sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)

    train_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    val_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

    train_dataset = HAM10000Dataset(metadata_df=train_df, image_dir=HAM_DIR, transforms=train_transform)
    val_dataset = HAM10000Dataset(metadata_df=val_df, image_dir=HAM_DIR, transforms=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename="peak-cbm-{epoch:02d}-{val_f2:.3f}",
        monitor="val_f2",
        mode="max",
        save_top_k=1,
    )

    early_stop_callback = EarlyStopping(
        monitor="val_f2",
        min_delta=0.005,
        patience=3,
        verbose=True,
        mode="max"
    )

    logger = TensorBoardLogger("/content/drive/MyDrive/XAI_Cloud_Run/tb_logs", name="GatedHybridCBM_Peak")

    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="auto",
        devices="auto",
        logger=logger,
        callbacks=[checkpoint_callback, early_stop_callback],
        precision="16-mixed",
        log_every_n_steps=10
    )

    print("Executing Native Boundary Shift...")
    trainer.fit(system, train_dataloaders=train_loader, val_dataloaders=val_loader)
    print(f"Peak Model finalized. Saved at: {checkpoint_callback.best_model_path}")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/XAI_Cloud_Run/finetune_peak.py


In [22]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python finetune_peak.py

Streaming output truncated to the last 5000 lines.
                                                               val_f2: 0.761    
                                                               val_recall: 0.816
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 86/501 0:00:18 • 0:01:25 4.90it/s v_num: 1.000     
                                                               train_loss: 0.021
                                                               val_f2: 0.761    
                                                               val_recall: 0.816
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 87/501 0:00:18 • 0:01:26 4.82it/s v_num: 1.000     
                                                               train_loss: 0.075
                                                               val_f2: 0.761    
                                                               val_recall: 0.816
Epoch 2/9  ━━╸━━━━━━━━━━━━━━ 88/501 0:00:18 • 0:01:26 4.82it/s v_num: 1.000     
                                                          

In [23]:
%%writefile /content/drive/MyDrive/XAI_Cloud_Run/peak_acid_test.py
import os
import torch
import pandas as pd
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from config import Config
from models import GatedHybridCBM
from engine import CBM_System

def execute_peak_audit():
    print("INITIALIZING PEAK DUAL-COHORT ACID TEST...")

    BASE_DIR = "/content/drive/MyDrive/XAI_Cloud_Run/"
    HAM_DIR = os.path.join(BASE_DIR, "HAM10000")
    ISIC_DIR = os.path.join(BASE_DIR, "isic_2020")
    CHECKPOINT_DIR = os.path.join(BASE_DIR, "cloud_checkpoints")

    # Locate the newly generated peak checkpoint
    peak_ckpts = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith('peak-cbm-')]
    if not peak_ckpts:
        print("ERROR: Peak checkpoint not found.")
        return
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, sorted(peak_ckpts)[-1])

    print(f"Injecting Peak Memory Overlay: {os.path.basename(CKPT_PATH)}...")
    base_model = GatedHybridCBM(
        concept_dims=Config.get_active_concept_dims(),
        backbone=Config.BACKBONE,
        feat_proj_dim=Config.FEAT_PROJ_DIM,
        hidden=Config.HIDDEN_DIM,
        dropout=Config.DROPOUT
    )

    system = CBM_System.load_from_checkpoint(CKPT_PATH, model=base_model, config=Config)
    system.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    system.to(device)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    print("\nPrepping Cohort A: HAM10000 Validation Sample...")
    ham_meta = [f for f in os.listdir(HAM_DIR) if f.endswith('.csv')][0]
    ham_df = pd.read_csv(os.path.join(HAM_DIR, ham_meta))

    binary_labels = (ham_df['dx'].astype(str).str.lower().str.strip() == 'mel').astype(int)
    _, val_df = train_test_split(ham_df, test_size=0.2, stratify=binary_labels, random_state=42)

    ham_sample = val_df.sample(n=200, random_state=42).reset_index(drop=True)
    ham_results = {'y_true': [], 'y_prob': [], 'alpha': []}

    with torch.no_grad():
        for _, row in tqdm(ham_sample.iterrows(), total=len(ham_sample), desc="Auditing HAM10k"):
            img_id = str(row['image_id'])
            img_path = None
            for root, _, files in os.walk(HAM_DIR):
                if f"{img_id}.jpg" in files:
                    img_path = os.path.join(root, f"{img_id}.jpg")
                    break

            if img_path:
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                ham_results['y_true'].append(1 if str(row['dx']).lower().strip() == 'mel' else 0)
                ham_results['y_prob'].append(torch.sigmoid(y_logit).item())
                ham_results['alpha'].append(alpha.item())

    print("\nPrepping Cohort B: ISIC 2020 Micro-Batch...")
    isic_csv = os.path.join(ISIC_DIR, "train.csv")
    isic_img_dir = os.path.join(ISIC_DIR, "images")

    if os.path.exists(isic_csv) and os.path.exists(isic_img_dir):
        isic_df = pd.read_csv(isic_csv)
        uploaded_files = [f for f in os.listdir(isic_img_dir) if f.endswith(('.jpg', '.jpeg'))]
        uploaded_basenames = [os.path.splitext(f)[0] for f in uploaded_files]
        test_df = isic_df[isic_df['image_name'].isin(uploaded_basenames)].reset_index(drop=True)

        isic_results = {'y_true': [], 'y_prob': [], 'alpha': []}

        with torch.no_grad():
            for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Auditing ISIC"):
                img_path = os.path.join(isic_img_dir, str(row['image_name']) + ".jpg")
                img_tensor = transform(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
                _, y_logit, alpha = system(img_tensor)
                isic_results['y_true'].append(row['target'])
                isic_results['y_prob'].append(torch.sigmoid(y_logit).item())
                isic_results['alpha'].append(alpha.item())
    else:
        print("WARNING: ISIC Data not found. Skipping Cohort B.")
        isic_results = None

    def print_report(name, res):
        print(f"\n" + "="*50)
        print(f"{name} AUDIT REPORT (PEAK FORM)")
        print("="*50)

        y_true = res['y_true']
        y_prob = res['y_prob']
        y_pred = [1 if p > 0.5 else 0 for p in y_prob]
        alphas = res['alpha']

        if len(set(y_true)) > 1:
            print(f"AUC Score: {roc_auc_score(y_true, y_prob):.3f}")

        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel() if len(cm.ravel()) == 4 else (0,0,0,0)

        print(f"\n--- NATIVE METRICS (Threshold 0.5) ---")
        print(f"Total Melanomas Present: {tp + fn}")
        print(f"FALSE NEGATIVES (Missed Cancers): {fn}")
        print(f"Sensitivity (Recall): {tp/(tp+fn) if (tp+fn)>0 else 0:.2f}")

        print(f"\n--- XAI INTEGRITY ---")
        print(f"Average Alpha Gate: {np.mean(alphas):.3f}")
        print(f"Black-Box Bias (<0.2): {sum(1 for a in alphas if a < 0.2) / len(alphas) * 100:.1f}%")

    print_report("COHORT A: HAM10000", ham_results)
    if isic_results:
        print_report("COHORT B: ISIC 2020", isic_results)

if __name__ == "__main__":
    execute_peak_audit()

Writing /content/drive/MyDrive/XAI_Cloud_Run/peak_acid_test.py


In [24]:
%cd /content/drive/MyDrive/XAI_Cloud_Run/
!python peak_acid_test.py

/content/drive/MyDrive/XAI_Cloud_Run
INITIALIZING PEAK DUAL-COHORT ACID TEST...
Injecting Peak Memory Overlay: peak-cbm-epoch=00-val_f2=0.769.ckpt...

Prepping Cohort A: HAM10000 Validation Sample...
Auditing HAM10k: 100% 200/200 [00:25<00:00,  7.86it/s]

Prepping Cohort B: ISIC 2020 Micro-Batch...
Auditing ISIC: 100% 27/27 [00:06<00:00,  4.47it/s]

COHORT A: HAM10000 AUDIT REPORT (PEAK FORM)
AUC Score: 0.990

--- NATIVE METRICS (Threshold 0.5) ---
Total Melanomas Present: 17
FALSE NEGATIVES (Missed Cancers): 1
Sensitivity (Recall): 0.94

--- XAI INTEGRITY ---
Average Alpha Gate: 0.660
Black-Box Bias (<0.2): 6.0%

COHORT B: ISIC 2020 AUDIT REPORT (PEAK FORM)

--- NATIVE METRICS (Threshold 0.5) ---
Total Melanomas Present: 0
FALSE NEGATIVES (Missed Cancers): 0
Sensitivity (Recall): 0.00

--- XAI INTEGRITY ---
Average Alpha Gate: 0.729
Black-Box Bias (<0.2): 0.0%
